# BoardMind AI: Whiteboard Detection Error Analysis & Failure Mining
**Notebook:** `07_whiteboard_error_analysis.ipynb`  
**Author:** Senior Computer Vision & ML Research Engineer  
**Hardware:** Intel Core i9-13900H CPU, 16GB RAM, Windows  

### Objectives:
1. Go beyond aggregate metrics to systematically identify failure modes:
   - **False Positives (FP):** Classroom walls, projector screens, door frames, or posters misclassified as whiteboards.
   - **False Negatives (FN):** Whiteboards missed due to extreme camera angle, heavy text, or severe occlusion.
   - **Localization Errors:** Inaccurate boundary detection (IoU < 0.75).
2. Mine representative failure cases and save visualizations to `runs/whiteboard_optimization/error_analysis/`.
3. Evaluate model robustness on challenging real-world test images (`board-srmap.jpg`).
4. Formulate concrete, targeted architectural and data recommendations.

In [ ]:
import sys
import os
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

ERROR_ANALYSIS_DIR = PROJECT_ROOT / 'runs' / 'whiteboard_optimization' / 'error_analysis'
ERROR_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Failure case output folder: {ERROR_ANALYSIS_DIR}")

## 1. Load Baseline and Optimized Checkpoints for Qualitative Comparison

In [ ]:
baseline_ckpt = PROJECT_ROOT / 'runs' / 'whiteboard_detection_test' / 'weights' / 'best.pt'
opt_ckpt = PROJECT_ROOT / 'models' / 'whiteboard_detector_best.pt'

if not opt_ckpt.exists():
    # Fall back to best available in optimization runs
    opt_candidates = list((PROJECT_ROOT / 'runs' / 'whiteboard_optimization').rglob('best.pt'))
    opt_ckpt = opt_candidates[0] if opt_candidates else baseline_ckpt

print(f"Comparing Baseline ({baseline_ckpt.name}) vs Optimized ({opt_ckpt})")
model_baseline = YOLO(str(baseline_ckpt))
model_optimized = YOLO(str(opt_ckpt))

## 2. Real-World Classroom Inference Test (`board-srmap.jpg`)
Testing on an independent real-world classroom photo to observe perspective distortion and boundary localization.

In [ ]:
test_img_path = PROJECT_ROOT / 'notebooks' / 'board-srmap.jpg'
print("Test image exists:", test_img_path.exists())

res_base = model_baseline.predict(str(test_img_path), conf=0.25, imgsz=416, device='cpu')[0]
res_opt = model_optimized.predict(str(test_img_path), conf=0.25, imgsz=416, device='cpu')[0]

# Render side-by-side
im_base = res_base.plot()
im_opt = res_opt.plot()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].imshow(cv2.cvtColor(im_base, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Baseline Detection (Boxes: {len(res_base.boxes)})")
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(im_opt, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"Optimized Model Detection (Boxes: {len(res_opt.boxes)})")
axes[1].axis('off')

plt.tight_layout()
plt.savefig(str(ERROR_ANALYSIS_DIR / 'comparison_board_srmap.jpg'))
plt.show()

## 3. Systematic Failure Case Mining on Validation Images
We scan validation images to identify cases where IoU with ground truth is low or confidence is near threshold.

In [ ]:
val_dir = PROJECT_ROOT / 'datasets' / 'whiteboard' / 'valid' / 'images'
val_labels = PROJECT_ROOT / 'datasets' / 'whiteboard' / 'valid' / 'labels'

val_imgs = sorted(list(val_dir.glob('*.*')))
print(f"Scanning {len(val_imgs)} validation images for edge cases...")

low_conf_cases = []
for img_p in val_imgs[:50]:
    res = model_optimized.predict(str(img_p), conf=0.1, imgsz=416, device='cpu', verbose=False)[0]
    if len(res.boxes) == 0:
        low_conf_cases.append((img_p, 'False Negative', 0.0))
    elif float(res.boxes.conf.min()) < 0.6:
        low_conf_cases.append((img_p, 'Low Confidence Detection', float(res.boxes.conf.min())))

print(f"Identified {len(low_conf_cases)} borderline/failure cases.")
for p, reason, conf in low_conf_cases[:5]:
    print(f"  {p.name}: {reason} (conf={conf:.2f})")